# 32. 현대 비전 모델 정리

이 노트북은 1장부터 4장까지의 흐름을 마무리하면서, ResNet, YOLO, U-Net, Mask R-CNN, ViT, DETR, Swin Transformer, SAM의 위치를 정리합니다.

중요한 것은 모델 이름을 외우는 것이 아니라, 각 모델이 어떤 문제를 풀기 위해 등장했는지 이해하는 것입니다.

이번 노트북의 목표는 다음과 같습니다.

- 이미지 분류, 객체 탐지, segmentation, foundation model의 차이를 정리합니다.
- CNN 계열과 Transformer 계열의 관점을 비교합니다.
- 대표 모델들이 전체 비전 파이프라인에서 어디에 위치하는지 파악합니다.
- 문제 상황에 따라 어떤 모델 계열을 먼저 고려할지 판단 기준을 세웁니다.

## 32-1. 준비

정리용 표와 시각화를 만들기 위해 간단한 Python 자료구조를 사용합니다.

In [ ]:
import numpy as np
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt

available_fonts = {f.name for f in fm.fontManager.ttflist}
for font_name in ['Malgun Gothic', 'AppleGothic', 'NanumGothic']:
    if font_name in available_fonts:
        plt.rcParams['font.family'] = font_name
        break

plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['axes.unicode_minus'] = False

## 32-2. 전체 문제 흐름

비전 모델의 흐름은 출력이 점점 더 풍부해지는 방향으로 이해할 수 있습니다.

```text
classification: 이미지가 무엇인가?
detection: 무엇이 어디에 있는가?
segmentation: 객체 영역은 어디까지인가?
promptable/foundation model: 사용자가 지정한 대상의 영역은 무엇인가?
```

In [ ]:
tasks = ['Classification', 'Detection', 'Segmentation', 'Promptable\nSegmentation']
outputs = ['class label', 'box + class', 'pixel mask', 'prompt -> mask']
models = ['ResNet / ViT', 'YOLO / DETR', 'U-Net / Mask R-CNN', 'SAM']
colors = ['#dbeafe', '#dcfce7', '#fef3c7', '#fee2e2']

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.axis('off')
ax.set_title('비전 문제의 출력 형태 변화')

for i, (task, output, model, color) in enumerate(zip(tasks, outputs, models, colors)):
    x = i * 2.4
    ax.add_patch(plt.Rectangle((x, 1.0), 1.9, 1.0, facecolor=color, edgecolor='#334155', linewidth=1.5))
    ax.text(x + 0.95, 1.72, task, ha='center', va='center', weight='bold')
    ax.text(x + 0.95, 1.42, output, ha='center', va='center', fontsize=9)
    ax.text(x + 0.95, 1.15, model, ha='center', va='center', fontsize=9)
    if i < len(tasks) - 1:
        ax.annotate('', xy=(x + 2.25, 1.5), xytext=(x + 1.95, 1.5), arrowprops=dict(arrowstyle='->'))

ax.set_xlim(-0.2, 9.4)
ax.set_ylim(0.6, 2.3)
plt.show()

## 32-3. 대표 모델의 위치

| 모델 | 주된 문제 | 핵심 아이디어 | 출력 |
|---|---|---|---|
| ResNet | 이미지 분류 | residual connection으로 깊은 CNN 학습 | class score |
| YOLO | 객체 탐지 | one-stage dense prediction | box, class, score |
| U-Net | semantic segmentation | encoder-decoder와 skip connection | pixel class map |
| Mask R-CNN | instance segmentation | detection에 mask branch 추가 | instance별 box, class, mask |
| ViT | 이미지 분류 | image patch를 token sequence로 처리 | class score |
| DETR | 객체 탐지 | set prediction과 object query | fixed-size prediction set |
| Swin Transformer | general vision backbone | window attention과 계층 구조 | multi-scale feature |
| SAM | promptable segmentation | prompt를 받아 mask 생성 | mask 후보, score |

In [ ]:
model_info = [
    ('ResNet', 'CNN', 'classification'),
    ('YOLO', 'CNN', 'detection'),
    ('U-Net', 'CNN', 'segmentation'),
    ('Mask R-CNN', 'CNN', 'instance segmentation'),
    ('ViT', 'Transformer', 'classification'),
    ('DETR', 'Transformer', 'detection'),
    ('Swin', 'Transformer', 'backbone'),
    ('SAM', 'Foundation', 'promptable segmentation'),
]

families = ['CNN', 'Transformer', 'Foundation']
task_order = ['classification', 'detection', 'segmentation', 'instance segmentation', 'backbone', 'promptable segmentation']

x_lookup = {task: i for i, task in enumerate(task_order)}
y_lookup = {family: i for i, family in enumerate(families)}
color_lookup = {'CNN': '#3b82f6', 'Transformer': '#16a34a', 'Foundation': '#ef4444'}

fig, ax = plt.subplots(figsize=(10, 4))
for name, family, task in model_info:
    x = x_lookup[task]
    y = y_lookup[family]
    ax.scatter(x, y, s=450, color=color_lookup[family], alpha=0.85)
    ax.text(x, y, name, color='white', ha='center', va='center', weight='bold', fontsize=9)

ax.set_xticks(range(len(task_order)))
ax.set_xticklabels(task_order, rotation=25, ha='right')
ax.set_yticks(range(len(families)))
ax.set_yticklabels(families)
ax.set_title('대표 모델의 계열과 사용 위치')
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

## 32-4. CNN 계열과 Transformer 계열 비교

| 관점 | CNN 계열 | Transformer 계열 |
|---|---|---|
| 기본 단위 | pixel grid, feature map | token sequence |
| 핵심 연산 | convolution | self-attention |
| 강한 가정 | local pattern, translation equivariance | token 관계 학습 |
| 장점 | 데이터 효율, local feature에 강함 | global relation, scale-up에 강함 |
| 대표 모델 | ResNet, YOLO, U-Net, Mask R-CNN | ViT, DETR, Swin |

현대 모델은 둘 중 하나만 고집하지 않는 경우가 많습니다. DETR은 CNN backbone과 Transformer decoder를 함께 쓰고, Swin은 Transformer이지만 CNN처럼 계층적 feature map을 만듭니다.

## 32-5. 문제별 모델 선택 기준

아래 기준은 시작점을 잡기 위한 실전적인 요약입니다.

In [ ]:
def recommend_model(task, need_instance=False, need_prompt=False, need_speed=False):
    if need_prompt:
        return 'SAM 계열을 먼저 검토합니다. class label이 필요하면 detector/classifier와 결합합니다.'
    if task == 'classification':
        return '데이터가 작거나 baseline이 필요하면 ResNet, 큰 데이터와 사전학습 활용이면 ViT도 검토합니다.'
    if task == 'detection':
        if need_speed:
            return '실시간성이 중요하면 YOLO 계열을 먼저 검토합니다.'
        return 'end-to-end set prediction 관점이 필요하면 DETR 계열도 검토합니다.'
    if task == 'segmentation':
        if need_instance:
            return '개별 객체 분리가 필요하면 Mask R-CNN 또는 instance segmentation 계열을 검토합니다.'
        return '클래스별 픽셀 지도가 필요하면 U-Net, DeepLabV3 같은 semantic segmentation 모델을 검토합니다.'
    return '먼저 출력 형태가 class, box, mask, prompt-mask 중 무엇인지 정의합니다.'

cases = [
    ('classification', False, False, False),
    ('detection', False, False, True),
    ('detection', False, False, False),
    ('segmentation', False, False, False),
    ('segmentation', True, False, False),
    ('segmentation', False, True, False),
]

for case in cases:
    print(case, '->', recommend_model(*case))

## 32-6. 전체 커리큘럼 관점에서 다시 보기

```text
1장: 이미지 분류
  ANN -> CNN -> ResNet -> Transfer Learning -> Grad-CAM

2장: 객체 탐지
  classification -> bbox -> IoU/NMS/mAP -> YOLO

3장: Segmentation
  detection -> semantic segmentation -> FCN -> U-Net -> Mask R-CNN

4장: Transformer 기반 비전 모델
  CNN의 한계 -> attention -> ViT -> DETR -> Swin -> SAM
```

이 순서로 보면 각 장은 이전 장의 출력 표현을 더 풍부하게 확장합니다. 분류는 class, 탐지는 box, segmentation은 mask, SAM은 prompt에 따른 mask 생성으로 이어집니다.

## 32-7. 최종 정리

- ResNet은 CNN 기반 이미지 분류의 강력한 기준점입니다.
- YOLO는 빠른 one-stage 객체 탐지 모델의 대표 흐름입니다.
- U-Net과 Mask R-CNN은 pixel mask를 다루는 segmentation 문제로 확장합니다.
- ViT는 이미지를 patch token sequence로 바꿔 Transformer에 넣습니다.
- DETR은 객체 탐지를 set prediction 문제로 재해석합니다.
- Swin은 window attention과 계층 구조로 Transformer backbone을 현실적인 비전 task에 연결합니다.
- SAM은 prompt를 통해 다양한 객체 영역을 mask로 추출하는 foundation segmentation 모델입니다.

이제 새로운 비전 모델을 볼 때는 `입력`, `출력`, `핵심 연산`, `후처리`, `어떤 문제를 해결하려는가`를 기준으로 위치를 잡으면 됩니다.